# Demo: identifisering av kontraktsklausuler
Sammenligning av søk og LLM mot CUAD-annotasjoner. **Kjør cellene ovenfra.**

Standardmodus ber om din egen API-nøkkel og gjør fire nye modellkall. Søk, sitatkontroll, sammenslåing og evaluering kjøres på nytt. Nøkkelen lagres ikke i filer.

Utvalget er illustrativt: to korte testkontrakter er valgt etter den endelige evalueringen for å vise både et funn søk overser og et falskt positivt LLM-funn. Det er ikke et representativt testsett, og målene her erstatter ikke rapportens resultater på 102 kontrakter. Ingen prompter er justert for demoen.

Det finnes ingen positive Post-Termination Services-annotasjoner i dette lille utvalget; F1 for kategorien her skal ikke tolkes som representativ ytelse.

## 1. Oppsett
Standardvalget `MODE = "api"` utfører fire nye modellkall med samme prompter som hovedforsøket. Lim inn egen nøkkel når feltet vises. `"lagret"` er et valgfritt alternativ som gjenbruker tidligere svar.

In [1]:
import json, time
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
import html
import demo_core as core

MODE = "api"  # Alternativ: "lagret" for tidligere modellsvar
if MODE not in ("lagret", "api"):
    raise ValueError("Velg lagret eller api.")
core.MODE = MODE
core.CALLS = 0
core.client = None
if MODE == "api":
    from getpass import getpass
    from openai import OpenAI
    key = getpass("Din OpenAI API-nøkkel: ").strip()
    try:
        if not key:
            raise ValueError("Nøkkelen er tom.")
        core.client = OpenAI(api_key=key, timeout=60, max_retries=0)
    finally:
        del key

documents = json.loads((core.BASE / "data/demo_kontrakter.json").read_text(encoding="utf-8"))
print("Modus:", MODE)
print("Modell i forsøket:", core.MODEL)
print("Kontrakter:", len(documents), "| Kategorier:", len(core.CATS))
print("Antall nye API-kall:", core.CONFIG["demo_request_count"] if MODE == "api" else 0)

Modus: api
Modell i forsøket: gpt-6-luna
Kontrakter: 3 | Kategorier: 2
Antall nye API-kall: 6


## 2. Data og avgrensning
Modellen får bare kontraktstekst og de fastlagte instruksjonene, ikke fasiten for demokontraktene. Few-shot-eksemplene i promptene stammer fra reserveutvalget.

In [2]:
display(pd.DataFrame([
    {"Kontrakt": d["title"], "Tegn": len(d["text_llm"]),
     "Tekstbiter": len(core.make_chunks(d["text_llm"]))}
    for d in documents
]))
for cat in core.CATS:
    print(cat + ":", core.DESCRIPTIONS[cat])

,Kontrakt,Tegn,Tekstbiter
0,WatchitMediaInc_20061201_8-K_EX-10.1_4148672_E...,2092,1
1,MphaseTechnologiesInc_20030911_10-K_EX-10.15_1...,7411,1
2,"PAXMEDICA,INC_07_02_2020-EX-10.12-Master Servi...",16768,1


Expiration Date: Identify the passage specifying when the agreement's initial term expires or how long it lasts. This may be an explicit calendar date, a duration measured from the effective or commencement date, or a statement that the agreement continues indefinitely, perpetually, or until terminated by notice. A calendar date is not required: a stated initial-term duration is sufficient. Return the original passage, not a calculated date. Exclude durations that apply only to individual obligations, such as confidentiality or payment, rather than to the agreement itself.
Post-Termination Services: Is a party subject to obligations after the termination or expiration of a contract, including any post-termination transition, payment, transfer of IP, wind-down, last-buy, or similar commitments?


## 3. Kjør søk og LLM-metoden
I lagret modus leses råsvarene via samme SHA-256-nøkkel som i hovedforsøket. Utdrag må finnes i originalteksten; utelatte sidemarkører på egne linjer tillates.

In [3]:
start = time.perf_counter()
details = {
    "søk": core.collect_predictions(core.keyword_predict, documents, core.CATS),
    "LLM": core.collect_predictions(core.llm_predict, documents, core.CATS),
}
print(f"Behandlingstid: {time.perf_counter() - start:.2f} sekunder")

Behandler: Expiration Date
Behandler: Post-Termination Services
Behandler: Expiration Date
Behandler: Post-Termination Services
Behandlingstid: 12.99 sekunder


## 4. Sammenlign utdrag med fasiten
Tom fasit betyr ingen annotert passasje for kategorien, ikke nødvendigvis at teksten er irrelevant ved juridisk gjennomgang. TP/FP/FN/TN her gjelder forekomst på kontraktsnivå.

In [4]:
def show_passages(label, passages):
    content = "".join("<li>" + html.escape(p) + "</li>" for p in passages)
    display(HTML("<b>" + html.escape(label) + "</b><ul>" +
                 (content or "<li>Ingen passasjer</li>") + "</ul>"))

for d in documents:
    for cat in core.CATS:
        display(HTML("<h3>" + html.escape(cat) + "</h3><p>" + html.escape(d["title"]) + "</p>"))
        show_passages("CUAD-fasit", [g["text"] for g in d["gold"][cat]])
        for method, records in details.items():
            r = next(r for r in records if r["title"] == d["title"] and r["kategori"] == cat)
            outcome = ("TP" if r["gold"] else "FP") if r["pred"] else ("FN" if r["gold"] else "TN")
            show_passages(method + " – " + outcome, r["pred"])

## 5. Evaluering og lagring
F1 måler klausulforekomst. `dekning_fasit` og `treffsikkerhet_utdrag` er gjennomsnittlig beste token-F1 per henholdsvis fasitpassasje og utdrag, med null for manglende treff. De er ikke vanlig recall/presisjon. Flere passasjer kan matches mot samme passasje.

In [5]:
results = pd.concat([
    core.evaluate_saved(records, core.CATS, method)
    for method, records in details.items()
], ignore_index=True)
display(results.round(3))
output = core.BASE / "output"
output.mkdir(exist_ok=True)
results.to_csv(output / (MODE + "_resultater.csv"), index=False)
(output / (MODE + "_prediksjoner.json")).write_text(
    json.dumps(details, ensure_ascii=False, indent=2), encoding="utf-8"
)
if MODE == "lagret":
    expected = json.loads((core.BASE / "data/forventede_prediksjoner.json").read_text(encoding="utf-8"))
    assert details == expected, "Resultatet avviker fra den validerte demoen."
    print("Kontroll OK: identiske prediksjoner med den validerte demokjøringen.")
print("Resultater lagret i:", output.resolve())

,metode,kategori,antall_kontrakter,TP,FP,FN,TN,presisjon,recall,F1,antall_fasitpassasjer,antall_utdrag,dekning_fasit,treffsikkerhet_utdrag
0,søk,Expiration Date,3,2,0,1,0,1.000,0.667,0.800,3,2,0.168,0.252
1,søk,Post-Termination Services,3,1,1,0,1,0.500,1.000,0.667,1,2,0.163,0.081
2,LLM,Expiration Date,3,3,0,0,0,1.000,1.000,1.000,3,3,0.851,0.851
3,LLM,Post-Termination Services,3,1,2,0,0,0.333,1.000,0.500,1,6,0.990,0.345


Resultater lagret i: C:\Users\fredr\Documents\GitHub\llmProject\demo\output


## 6. Hva viser hovedforsøket?
På hele testsettet (102 kontrakter, seks kategorier) var makro-F1 **0,643 for søk** og **0,784 for LLM**. Parret bootstrap ga 95 % intervall **[0,086, 0,203]** for forskjellen LLM − søk. Disse er tidligere beregnede rapportresultater, ikke beregnet fra demoens utvalg.

LLM var best i fire kategorier, mens søk hadde høyere F1 for IP Ownership Assignment og Post-Termination Services. Demoens falske positive illustrerer hvorfor høy gjenfinning ikke er tilstrekkelig. Tomme annotasjoner og brede kategoriavgrensninger må diskuteres, uten å endre fasiten.

Datakilde: CUAD / The Atticus Project. Koden er tilpasset gruppens M1.ipynb med KI-assistanse.